In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# timezone for UTC+8
sh = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_log = pd.read_csv('log.csv')
df_trace = pd.read_csv('trace.csv')
df_errors = pd.read_csv('error_logs.csv')

# Ensure value columns are numeric where applicable
for df in (df_metric, df_log, df_trace):
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

# Helper to aggregate by (cmdb_id, KPI)
def aggregate_kpi(df, kpi_col, source_name):
    # full group aggregations
    agg_basic = df.groupby(['cmdb_id', kpi_col])['value'].agg(['count','min','max','mean']).reset_index()
    # p95 computed globally per series (entire series as required)
    p95 = df.groupby(['cmdb_id', kpi_col])['value'].quantile(0.95).reset_index(name='p95')
    agg = agg_basic.merge(p95, on=['cmdb_id', kpi_col])
    # normalize column names
    agg = agg.rename(columns={kpi_col: 'kpi_name'})
    agg['source_file'] = source_name
    # reorder columns
    agg = agg[['source_file','cmdb_id','kpi_name','count','p95','min','max','mean']]
    return agg

# Aggregate each file and keep full results in variables for next steps
metric_agg_full = aggregate_kpi(df_metric, 'kpi_name', 'metric.csv')
log_agg_full = aggregate_kpi(df_log, 'log_name', 'log.csv')
trace_agg_full = aggregate_kpi(df_trace, 'trace_name', 'trace.csv')

# Prepare compact tables: top 50 rows sorted by p95 desc
metric_compact = metric_agg_full.sort_values(by='p95', ascending=False).head(50).reset_index(drop=True)
log_compact = log_agg_full.sort_values(by='p95', ascending=False).head(50).reset_index(drop=True)
trace_compact = trace_agg_full.sort_values(by='p95', ascending=False).head(50).reset_index(drop=True)

# Error logs summary
error_total_count = len(df_errors)
error_counts_per_cmdb = (
    df_errors.groupby('cmdb_id').size().reset_index(name='count')
    .sort_values(by='count', ascending=False)
)
error_counts_top20 = error_counts_per_cmdb.head(20).reset_index(drop=True)

# earliest 5 rows with timestamp converted to Asia/Shanghai (UTC+8)
if 'timestamp' in df_errors.columns:
    earliest5 = df_errors.sort_values('timestamp').head(5).copy()
    # convert epoch seconds to timezone-aware datetime in Asia/Shanghai
    earliest5['ts_utc8'] = pd.to_datetime(earliest5['timestamp'], unit='s', utc=True).dt.tz_convert(sh)
    earliest5_small = earliest5[['ts_utc8','cmdb_id','message']].reset_index(drop=True)
else:
    earliest5_small = df_errors.head(5)[['cmdb_id','message']].reset_index(drop=True)

# Display compact outputs (variables). Keep outputs concise as required.
metric_compact, log_compact, trace_compact, error_total_count, error_counts_top20, earliest5_small

```
Out[1]:
```
summary = (
    "Summary of telemetry aggregation (full-series statistics used):\n"
    "- metric.csv: The largest P95 values are dominated by Redis metrics (Redis01/Redis02): high network packet counters (NETPacketsIn/Out) and Redis metrics (total_commands_processed and maxmemory ~4.29e9). Tomcat request-processing metrics also show very large P95s (several 1e8–4e8 range). IG nodes (IG01/IG02) also surface high network packet P95s.\n"
    "- log.csv: log.row_count P95 is highest on apache02 (p95=1030) and apache01 (p95=740.5); multiple Tomcat instances (Tomcat04/03/01/02) show elevated log.row_count as well. GC pause maxima are small (~0.08–0.10). Error_count is mostly zero for Tomcats, small for apache nodes.\n"
    "- trace.csv: trace.self.row_count P95 is highest for dockerB1 (3774.65) and dockerB2 (3490.25), with Tomcat04/01/03 also high (~3k). Many inter-service trace row_count series have p95 values in the low hundreds (~231).\n"
    "- error_logs.csv: total rows = 52. Top cmdb_id counts: apache02 (26), apache01 (25), Tomcat04 (1). Earliest error log rows (first 5) show HTTP POST requests and IPAddress entries from Tomcat04, apache01, and apache02 involving /UOCP/... endpoints.\n\n"
    "Stored variables for follow-up: metric_agg_full, log_agg_full, trace_agg_full (full aggregated tables), and error_counts_top20 / earliest5_small (error summaries)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   source_file   cmdb_id                                           kpi_name  count           p95           min           max          mean
0   metric.csv   Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     16  5.307932e+09  3.065000e+03  5.309466e+09  1.989644e+09
1   metric.csv   Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut     16  4.804004e+09  2.886000e+03  4.805455e+09  1.800712e+09
2   metric.csv   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     31  4.674156e+09  4.668053e+09  4.674405e+09  4.671293e+09
3   metric.csv   Redis01                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
4   metric.csv   Redis02                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
..         ...       ...                                                ...    ...           ...           ...           ...           ...
45  metric.csv  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.773671e+08  4.714772e+08  4.774550e+08  4.746708e+08
46  metric.csv  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.558436e+08  4.502600e+08  4.559966e+08  4.532501e+08
47  metric.csv  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  4.472656e+08  4.415350e+08  4.473939e+08  4.446440e+08
48  metric.csv      IG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     16  3.838013e+08  1.970000e+02  3.838512e+08  1.678530e+08
49  metric.csv      IG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn     15  3.824558e+08  1.800000e+02  3.825088e+08  1.274605e+08

[50 rows x 8 columns],    source_file   cmdb_id          kpi_name  count      p95    min      max        mean
0      log.csv  apache02     log.row_count     31  1030.00  10.00  1082.00  880.580645
1      log.csv  apache01     log.row_count     31   740.50   4.00   747.00  689.903226
2      log.csv  Tomcat04     log.row_count     31   463.00   2.00   497.00  394.032258
3      log.csv  Tomcat03     log.row_count     31   453.50   3.00   459.00  389.774194
4      log.csv  Tomcat01     log.row_count     31   442.00   1.00   449.00  393.903226
5      log.csv  Tomcat02     log.row_count     31   434.50   5.00   458.00  390.419355
6      log.csv   Mysql01     log.row_count     23     4.00   1.00     8.00    2.260870
7      log.csv  Tomcat02  log.gc_pause_max     30     0.10   0.03     0.12    0.080333
8      log.csv  Tomcat01  log.gc_pause_max     21     0.10   0.02     0.10    0.071905
9      log.csv  Tomcat03  log.gc_pause_max     21     0.09   0.02     0.10    0.063810
10     log.csv  Tomcat04  log.gc_pause_max     21     0.08   0.02     0.08    0.063333
11     log.csv   Mysql01   log.error_count     23     0.00   0.00     0.00    0.000000
12     log.csv  Tomcat01   log.error_count     31     0.00   0.00     0.00    0.000000
13     log.csv  Tomcat02   log.error_count     31     0.00   0.00     0.00    0.000000
14     log.csv  Tomcat03   log.error_count     31     0.00   0.00     0.00    0.000000
15     log.csv  Tomcat04   log.error_count     31     0.00   0.00     1.00    0.032258
16     log.csv  apache01   log.error_count     31     0.00   0.00    25.00    0.806452
17     log.csv  apache02   log.error_count     31     0.00   0.00    26.00    0.838710,    source_file   cmdb_id                       kpi_name  count      p95     min     max         mean
0    trace.csv  dockerB1           trace.self.row_count     30  3774.65  1408.0  4256.0  2445.000000
1    trace.csv  dockerB2           trace.self.row_count     30  3490.25  1237.0  4057.0  2469.733333
2    trace.csv  Tomcat04           trace.self.row_count     30  3172.95   712.0  3502.0  2748.233333
3    trace.csv  Tomcat01           trace.self.row_count     30  3042.70  2120.0  3112.0  2744.100000
4    trace.csv  Tomcat03           trace.self.row_count     30  3014.10  2038.0  3060.0  2706.266667
..         ...       ...                            ...    ...      ...     ...     ...          ...
45   trace.csv      IG01    trace.to_Tomcat03.row_count     30   233.00   159.0   235.0   201.966667
46   trace.csv      MG01  trace.from_Tomcat01.row_count     30   231.30     2.0   235.0   166.133333
47   trace.csv  Tomcat01        trace.to_MG01.row_count     30   231.30     2.0   235.0   166.133333
48   trace.csv  Tomcat02      trace.from_IG02.row_count     30   231.15   132.0   237.0   200.133333
49   trace.csv      IG02    trace.to_Tomcat02.row_count     30   231.15   132.0   237.0   200.133333

[50 rows x 8 columns], 52,     cmdb_id  count
0  apache02     26
1  apache01     25
2  Tomcat04      1,                     ts_utc8   cmdb_id                                            message
0 2021-03-07 16:13:42+08:00  Tomcat04  IG02 POST /UOCP/person/ServiceTest9.json HTTP/...
1 2021-03-07 16:29:03+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso...
2 2021-03-07 16:29:03+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
3 2021-03-07 16:29:05+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
4 2021-03-07 16:29:06+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

sh = pytz.timezone('Asia/Shanghai')

# Reuse existing variables: df_metric, metric_agg_full
# Ensure numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
metric_agg_full['p95'] = pd.to_numeric(metric_agg_full['p95'], errors='coerce')

# Define window in UTC (convert from Asia/Shanghai to UTC for epoch comparison)
start_local = pd.Timestamp("2021-03-07 16:00:00").tz_localize(sh)
end_local = pd.Timestamp("2021-03-07 16:30:00").tz_localize(sh)
start_utc = start_local.tz_convert('UTC')
end_utc = end_local.tz_convert('UTC')

# Convert window to epoch seconds for filtering (metric timestamps are epoch seconds UTC)
start_epoch = int(start_utc.timestamp())
end_epoch = int(end_utc.timestamp())

# Filter metric rows within the window (use full-series p95 computed earlier)
metric_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()

# Convert timestamp to tz-aware ts_utc8 column
metric_window['ts_utc8'] = pd.to_datetime(metric_window['timestamp'], unit='s', utc=True).dt.tz_convert(sh)

# Join global p95 from metric_agg_full by (cmdb_id, kpi_name)
metric_joined_window = metric_window.merge(
    metric_agg_full[['cmdb_id','kpi_name','p95']],
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Flag anomaly if value >= p95 (and p95 not null). Compute severity = value / p95 (float)
metric_joined_window['is_anomaly'] = (~metric_joined_window['p95'].isna()) & (metric_joined_window['value'] >= metric_joined_window['p95'])
# Avoid division by zero: set severity to NaN when p95 <= 0 or NaN
metric_joined_window['severity'] = np.where(
    (metric_joined_window['p95'] > 0),
    metric_joined_window['value'] / metric_joined_window['p95'],
    np.nan
)

# Anomalies subset
anomalies = metric_joined_window[metric_joined_window['is_anomaly']].copy()

# a) Anomaly summary aggregated by (cmdb_id, kpi_name)
if not anomalies.empty:
    anomaly_summary = anomalies.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count=('is_anomaly','size'),
        earliest_anomaly_ts_utc8=('ts_utc8','min'),
        max_severity=('severity','max')
    ).reset_index()
    anomaly_summary_top20 = anomaly_summary.sort_values(by='max_severity', ascending=False).head(20).reset_index(drop=True)
else:
    anomaly_summary_top20 = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts_utc8','max_severity'])

# b) Sample anomaly rows table (top 20 by severity)
if not anomalies.empty:
    anomaly_sample_top20 = anomalies.sort_values(by='severity', ascending=False).head(20)[
        ['ts_utc8','cmdb_id','kpi_name','value','p95','severity']
    ].reset_index(drop=True)
else:
    anomaly_sample_top20 = pd.DataFrame(columns=['ts_utc8','cmdb_id','kpi_name','value','p95','severity'])

# Total anomaly count across all metrics in the window
total_anomaly_count = int(anomalies.shape[0])

# Display compact outputs
anomaly_summary_top20, anomaly_sample_top20, total_anomaly_count

```
Out[2]:
```
summary = (
    "Summary for window 2021-03-07 16:00–16:30 (UTC+8):\n"
    f"- Total metric anomaly rows detected: {total_anomaly_count}\n"
    "- Top anomaly groups by max severity:\n"
    f"  1) {anomaly_summary_top20.iloc[0]['cmdb_id']} - {anomaly_summary_top20.iloc[0]['kpi_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[0]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[0]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[0]['max_severity']}\n"
    f"  2) {anomaly_summary_top20.iloc[1]['cmdb_id']} - {anomaly_summary_top20.iloc[1]['kpi_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[1]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[1]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[1]['max_severity']}\n"
    f"  3) {anomaly_summary_top20.iloc[2]['cmdb_id']} - {anomaly_summary_top20.iloc[2]['kpi_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[2]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[2]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[2]['max_severity']}\n"
    "- Top sample anomaly rows (highest severity):\n"
    f"  * {anomaly_sample_top20.iloc[0]['ts_utc8']} | {anomaly_sample_top20.iloc[0]['cmdb_id']} | {anomaly_sample_top20.iloc[0]['kpi_name']} | "
    f"value={anomaly_sample_top20.iloc[0]['value']} | p95={anomaly_sample_top20.iloc[0]['p95']} | severity={anomaly_sample_top20.iloc[0]['severity']}\n"
    f"  * {anomaly_sample_top20.iloc[1]['ts_utc8']} | {anomaly_sample_top20.iloc[1]['cmdb_id']} | {anomaly_sample_top20.iloc[1]['kpi_name']} | "
    f"value={anomaly_sample_top20.iloc[1]['value']} | p95={anomaly_sample_top20.iloc[1]['p95']} | severity={anomaly_sample_top20.iloc[1]['severity']}\n"
    "  * Additional notable anomalies include several MySQL/disk metrics and CPU WIO on IG02 with elevated severities (~3–7 and higher).\n\n"
    "Actionable insight: investigate IG02 and Tomcat04 network TCP-FIN-WAIT spikes first (very high severity), then review MySQL/disk IO (Mysql02/Mysql01) and IG02 CPU WIO."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts_utc8  max_severity
0       IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              2 2021-03-07 16:21:00+08:00    124.000000
1   Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              2 2021-03-07 16:12:00+08:00     51.000000
2       MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              1 2021-03-07 16:04:00+08:00     20.000000
3       MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT              1 2021-03-07 16:13:00+08:00     20.000000
4       MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              1 2021-03-07 16:12:00+08:00     10.000000
5       IG02                             OSLinux-CPU_CPU_CPUWio              2 2021-03-07 16:02:00+08:00      7.043732
6    Mysql02               Mysql-MySQL_3306_Max trx rows locked              3 2021-03-07 16:06:00+08:00      7.000000
7    Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps              1 2021-03-07 16:00:00+08:00      4.000000
8    Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1 2021-03-07 16:22:00+08:00      4.000000
9    Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead              1 2021-03-07 16:00:00+08:00      4.000000
10   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1 2021-03-07 16:30:00+08:00      4.000000
11   Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time              2 2021-03-07 16:00:00+08:00      3.825773
12  Tomcat01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT              1 2021-03-07 16:13:00+08:00      3.428571
13   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1 2021-03-07 16:11:00+08:00      3.333333
14   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps              1 2021-03-07 16:23:00+08:00      3.333333
15  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps              1 2021-03-07 16:23:00+08:00      3.333333
16   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              1 2021-03-07 16:05:00+08:00      3.333333
17   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...              1 2021-03-07 16:15:00+08:00      3.333333
18      IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...              1 2021-03-07 16:29:00+08:00      3.333333
19      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps              1 2021-03-07 16:02:00+08:00      3.278006,                      ts_utc8   cmdb_id                                           kpi_name     value        p95    severity
0  2021-03-07 16:30:00+08:00      IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   62.0000   0.500000  124.000000
1  2021-03-07 16:30:00+08:00  Tomcat04       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   51.0000   1.000000   51.000000
2  2021-03-07 16:04:00+08:00      MG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    1.0000   0.050000   20.000000
3  2021-03-07 16:13:00+08:00      MG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    1.0000   0.050000   20.000000
4  2021-03-07 16:12:00+08:00      MG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT   19.0000   1.900000   10.000000
5  2021-03-07 16:02:00+08:00      IG02                             OSLinux-CPU_CPU_CPUWio    1.2887   0.182957    7.043732
6  2021-03-07 16:25:00+08:00   Mysql02               Mysql-MySQL_3306_Max trx rows locked    7.0000   1.000000    7.000000
7  2021-03-07 16:00:00+08:00   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRead  103.8667  25.966675    4.000000
8  2021-03-07 16:00:00+08:00   Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTps    1.1833   0.295825    4.000000
9  2021-03-07 16:22:00+08:00   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    3.2000   0.800000    4.000000
10 2021-03-07 16:30:00+08:00   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...   35.6000   8.900000    4.000000
11 2021-03-07 16:01:00+08:00   Mysql01              Mysql-MySQL_3306_Innodb Row Lock Time    8.4167   2.200000    3.825773
12 2021-03-07 16:13:00+08:00  Tomcat01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    6.0000   1.750000    3.428571
13 2021-03-07 16:23:00+08:00  apache02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    1.0000   0.300000    3.333333
14 2021-03-07 16:23:00+08:00   Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    5.0000   1.500000    3.333333
15 2021-03-07 16:29:00+08:00      IG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.2000   0.060000    3.333333
16 2021-03-07 16:11:00+08:00   Mysql02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    7.0000   2.100000    3.333333
17 2021-03-07 16:05:00+08:00   Mysql02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...   10.4000   3.120000    3.333333
18 2021-03-07 16:15:00+08:00   Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...    7.2000   2.160000    3.333333
19 2021-03-07 16:02:00+08:00      IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    6.5833   2.008325    3.278006, 15514)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone and existing variables: df_trace, trace_agg_full
sh = pytz.timezone('Asia/Shanghai')

# ensure numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
trace_agg_full['p95'] = pd.to_numeric(trace_agg_full['p95'], errors='coerce')

# define window in UTC based on Asia/Shanghai local times
start_local = pd.Timestamp("2021-03-07 16:00:00").tz_localize(sh)
end_local = pd.Timestamp("2021-03-07 16:30:00").tz_localize(sh)
start_utc = start_local.tz_convert('UTC')
end_utc = end_local.tz_convert('UTC')
start_epoch = int(start_utc.timestamp())
end_epoch = int(end_utc.timestamp())

# filter trace rows in the window
trace_window = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()

# convert timestamp to tz-aware Asia/Shanghai for display
trace_window['ts_utc8'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(sh)

# join global p95 from trace_agg_full (trace_agg_full kpi name column is 'kpi_name')
trace_joined_window = trace_window.merge(
    trace_agg_full[['cmdb_id','kpi_name','p95']],
    left_on=['cmdb_id','trace_name'],
    right_on=['cmdb_id','kpi_name'],
    how='left'
)

# flag anomalies and compute severity (avoid div by zero)
trace_joined_window['is_anomaly'] = (~trace_joined_window['p95'].isna()) & (trace_joined_window['value'] >= trace_joined_window['p95'])
trace_joined_window['severity'] = np.where(
    (trace_joined_window['p95'] > 0),
    trace_joined_window['value'] / trace_joined_window['p95'],
    np.nan
)

# extract anomalies
trace_anomalies = trace_joined_window[trace_joined_window['is_anomaly']].copy()

# aggregated anomaly summary by (cmdb_id, trace_name)
if not trace_anomalies.empty:
    anomaly_summary = trace_anomalies.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count=('is_anomaly','size'),
        earliest_anomaly_ts_utc8=('ts_utc8','min'),
        max_severity=('severity','max')
    ).reset_index()
    anomaly_summary_top20 = anomaly_summary.sort_values(by='max_severity', ascending=False).head(20).reset_index(drop=True)
else:
    anomaly_summary_top20 = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts_utc8','max_severity'])

# sample anomaly rows (top 20 by severity)
if not trace_anomalies.empty:
    anomaly_sample_top20 = trace_anomalies.sort_values(by='severity', ascending=False).head(20)[
        ['ts_utc8','cmdb_id','trace_name','value','p95','severity']
    ].reset_index(drop=True)
else:
    anomaly_sample_top20 = pd.DataFrame(columns=['ts_utc8','cmdb_id','trace_name','value','p95','severity'])

# total anomaly count in the window
total_trace_anomaly_count = int(trace_anomalies.shape[0])

# Display compact outputs (keep full joined table trace_joined_window in kernel)
anomaly_summary_top20, anomaly_sample_top20, total_trace_anomaly_count

```
Out[3]:
```
summary = (
    "Trace anomaly summary for 2021-03-07 16:00–16:30 (UTC+8):\n"
    f"- Total trace anomaly rows detected in window: {total_trace_anomaly_count}\n"
    "- Highest-severity anomaly groups to investigate first:\n"
    f"  1) {anomaly_summary_top20.iloc[0]['cmdb_id']} - {anomaly_summary_top20.iloc[0]['trace_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[0]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[0]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[0]['max_severity']:.2f}\n"
    f"  2) {anomaly_summary_top20.iloc[1]['cmdb_id']} - {anomaly_summary_top20.iloc[1]['trace_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[1]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[1]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[1]['max_severity']:.2f}\n"
    f"  3) {anomaly_summary_top20.iloc[2]['cmdb_id']} - {anomaly_summary_top20.iloc[2]['trace_name']}: "
    f"anomaly_count={anomaly_summary_top20.iloc[2]['anomaly_count']}, earliest={anomaly_summary_top20.iloc[2]['earliest_anomaly_ts_utc8']}, "
    f"max_severity={anomaly_summary_top20.iloc[2]['max_severity']:.2f}\n"
    "- Notes from sample anomaly rows (highest severity examples):\n"
    f"  * {anomaly_sample_top20.iloc[0]['ts_utc8']} | {anomaly_sample_top20.iloc[0]['cmdb_id']} | {anomaly_sample_top20.iloc[0]['trace_name']} | "
    f"severity={anomaly_sample_top20.iloc[0]['severity']:.2f}\n"
    f"  * {anomaly_sample_top20.iloc[1]['ts_utc8']} | {anomaly_sample_top20.iloc[1]['cmdb_id']} | {anomaly_sample_top20.iloc[1]['trace_name']} | "
    f"severity={anomaly_sample_top20.iloc[1]['severity']:.2f}\n"
    "- Immediate action suggestions: prioritize investigation of dockerA1 and IG01/IG02 service latency spikes (very high severity), then check MG components and inter-service paths showing 2–4x degradations. Correlate with CPU, network, and downstream service health during the listed timestamps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts_utc8  max_severity
0   dockerA1            trace.self.duration_p95              2 2021-03-07 16:24:00+08:00     57.059490
1       IG01            trace.self.duration_p95              2 2021-03-07 16:04:00+08:00     19.217803
2       IG02            trace.self.duration_p95              2 2021-03-07 16:04:00+08:00     19.188487
3       IG02           trace.self.duration_mean              2 2021-03-07 16:12:00+08:00      9.964907
4       IG01           trace.self.duration_mean              2 2021-03-07 16:12:00+08:00      9.640061
5   dockerA1        trace.to_MG01.duration_mean              2 2021-03-07 16:12:00+08:00      3.993377
6       MG01  trace.from_dockerA1.duration_mean              2 2021-03-07 16:12:00+08:00      3.993377
7   dockerA2        trace.to_MG01.duration_mean              2 2021-03-07 16:10:00+08:00      2.509099
8       MG01  trace.from_dockerA2.duration_mean              2 2021-03-07 16:10:00+08:00      2.509099
9       MG01    trace.to_dockerA1.duration_mean              2 2021-03-07 16:10:00+08:00      2.124622
10  dockerA1      trace.from_MG01.duration_mean              2 2021-03-07 16:10:00+08:00      2.124622
11  dockerB1           trace.self.duration_mean              2 2021-03-07 16:10:00+08:00      1.690649
12  Tomcat03        trace.to_MG01.duration_mean              2 2021-03-07 16:04:00+08:00      1.602074
13      MG01  trace.from_Tomcat03.duration_mean              2 2021-03-07 16:04:00+08:00      1.602074
14  dockerB2           trace.self.duration_mean              2 2021-03-07 16:09:00+08:00      1.599813
15  dockerA1        trace.to_MG02.duration_mean              2 2021-03-07 16:10:00+08:00      1.586324
16      MG02  trace.from_dockerA1.duration_mean              2 2021-03-07 16:10:00+08:00      1.586324
17      MG01  trace.from_Tomcat02.duration_mean              2 2021-03-07 16:04:00+08:00      1.499592
18  Tomcat02        trace.to_MG01.duration_mean              2 2021-03-07 16:04:00+08:00      1.499592
19      MG02    trace.to_dockerB2.duration_mean              2 2021-03-07 16:12:00+08:00      1.414647,                      ts_utc8   cmdb_id                         trace_name      value       p95   severity
0  2021-03-07 16:28:00+08:00  dockerA1            trace.self.duration_p95   1.007100  0.017650  57.059490
1  2021-03-07 16:29:00+08:00      IG01            trace.self.duration_p95  61.004850  3.174392  19.217803
2  2021-03-07 16:29:00+08:00      IG02            trace.self.duration_p95  60.000000  3.126875  19.188487
3  2021-03-07 16:29:00+08:00      IG02           trace.self.duration_mean   6.903000  0.692731   9.964907
4  2021-03-07 16:29:00+08:00      IG01           trace.self.duration_mean   6.826283  0.708116   9.640061
5  2021-03-07 16:12:00+08:00  dockerA1        trace.to_MG01.duration_mean   0.512500  0.128337   3.993377
6  2021-03-07 16:12:00+08:00      MG01  trace.from_dockerA1.duration_mean   0.512500  0.128337   3.993377
7  2021-03-07 16:10:00+08:00      MG01  trace.from_dockerA2.duration_mean   1.103000  0.439600   2.509099
8  2021-03-07 16:10:00+08:00  dockerA2        trace.to_MG01.duration_mean   1.103000  0.439600   2.509099
9  2021-03-07 16:10:00+08:00  dockerA1      trace.from_MG01.duration_mean   0.856400  0.403083   2.124622
10 2021-03-07 16:10:00+08:00      MG01    trace.to_dockerA1.duration_mean   0.856400  0.403083   2.124622
11 2021-03-07 16:12:00+08:00  dockerB1           trace.self.duration_mean   0.163450  0.096679   1.690649
12 2021-03-07 16:10:00+08:00  Tomcat03        trace.to_MG01.duration_mean   1.057200  0.659895   1.602074
13 2021-03-07 16:10:00+08:00      MG01  trace.from_Tomcat03.duration_mean   1.057200  0.659895   1.602074
14 2021-03-07 16:12:00+08:00  dockerB2           trace.self.duration_mean   0.116695  0.072943   1.599813
15 2021-03-07 16:10:00+08:00  dockerA1        trace.to_MG02.duration_mean   0.350667  0.221056   1.586324
16 2021-03-07 16:10:00+08:00      MG02  trace.from_dockerA1.duration_mean   0.350667  0.221056   1.586324
17 2021-03-07 16:09:00+08:00  Tomcat02        trace.to_MG01.duration_mean   0.795750  0.530644   1.499592
18 2021-03-07 16:09:00+08:00      MG01  trace.from_Tomcat02.duration_mean   0.795750  0.530644   1.499592
19 2021-03-07 16:12:00+08:00  dockerB2      trace.from_MG02.duration_mean   0.934165  0.660352   1.414647, 526)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse timezone variable if exists, else create
try:
    sh
except NameError:
    sh = pytz.timezone('Asia/Shanghai')

# Reuse existing loaded dataframes: df_errors, df_log, log_agg_full
# Ensure they exist; if not, load from CSV (safe fallback)
import os
if 'df_errors' not in globals():
    df_errors = pd.read_csv('error_logs.csv')
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')
if 'log_agg_full' not in globals():
    # try to compute log_agg_full if missing (uses full-series global p95)
    df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')
    def aggregate_kpi(df, kpi_col, source_name):
        agg_basic = df.groupby(['cmdb_id', kpi_col])['value'].agg(['count','min','max','mean']).reset_index()
        p95 = df.groupby(['cmdb_id', kpi_col])['value'].quantile(0.95).reset_index(name='p95')
        agg = agg_basic.merge(p95, on=['cmdb_id', kpi_col])
        agg = agg.rename(columns={kpi_col: 'kpi_name'})
        agg['source_file'] = source_name
        agg = agg[['source_file','cmdb_id','kpi_name','count','p95','min','max','mean']]
        return agg
    log_agg_full = aggregate_kpi(df_log, 'log_name', 'log.csv')

# Define window in Asia/Shanghai then convert to UTC epoch seconds
start_local = pd.Timestamp("2021-03-07 16:00:00").tz_localize(sh)
end_local = pd.Timestamp("2021-03-07 16:30:00").tz_localize(sh)
start_epoch = int(start_local.tz_convert('UTC').timestamp())
end_epoch = int(end_local.tz_convert('UTC').timestamp())

# ---- Part 1: error_logs filtering and summaries ----
# Ensure timestamp numeric
df_errors['timestamp'] = pd.to_numeric(df_errors['timestamp'], errors='coerce')

# Filter error logs in window
error_window = df_errors[(df_errors['timestamp'] >= start_epoch) & (df_errors['timestamp'] <= end_epoch)].copy()

# Convert timestamp to ts_utc8 for display
if 'timestamp' in error_window.columns:
    error_window['ts_utc8'] = pd.to_datetime(error_window['timestamp'], unit='s', utc=True).dt.tz_convert(sh)

# 1) total number of error_log rows in the window
error_window_count = int(error_window.shape[0])

# 2) counts per cmdb_id (top 20) sorted by count desc
error_counts_per_cmdb = error_window.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False).reset_index(drop=True)
error_counts_top20 = error_counts_per_cmdb.head(20)

# 3) earliest up to 20 rows in the window with columns: ts_utc8, cmdb_id, message
if 'ts_utc8' in error_window.columns:
    error_earliest20 = error_window.sort_values('timestamp').head(20)[['ts_utc8','cmdb_id','message']].reset_index(drop=True)
else:
    error_earliest20 = error_window.sort_values('timestamp').head(20)[['cmdb_id','message']].reset_index(drop=True)

# ---- Part 2: log.csv filtering for specified components and anomaly detection ----
# Components of interest
components = ['Tomcat04', 'IG02', 'apache01', 'apache02', 'Tomcat01']

# Ensure numeric value
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Filter log rows in window and components
log_window_filtered = df_log[
    (df_log['timestamp'] >= start_epoch) &
    (df_log['timestamp'] <= end_epoch) &
    (df_log['cmdb_id'].isin(components))
].copy()

# Convert timestamp to ts_utc8 for display
if 'timestamp' in log_window_filtered.columns:
    log_window_filtered['ts_utc8'] = pd.to_datetime(log_window_filtered['timestamp'], unit='s', utc=True).dt.tz_convert(sh)

# Join with global p95 from precomputed log_agg_full by (cmdb_id, log_name)
# log_agg_full uses column 'kpi_name' for the KPI name
log_joined_window = log_window_filtered.merge(
    log_agg_full[['cmdb_id','kpi_name','p95']],
    left_on=['cmdb_id','log_name'],
    right_on=['cmdb_id','kpi_name'],
    how='left'
)

# Flag anomalies and compute severity (avoid div by zero)
log_joined_window['is_anomaly'] = (~log_joined_window['p95'].isna()) & (log_joined_window['value'] >= log_joined_window['p95'])
log_joined_window['severity'] = np.where(
    (log_joined_window['p95'] > 0),
    log_joined_window['value'] / log_joined_window['p95'],
    np.nan
)

# Aggregate anomaly summary by (cmdb_id, log_name)
log_anomalies = log_joined_window[log_joined_window['is_anomaly']].copy()

if not log_anomalies.empty:
    log_anomaly_summary = log_anomalies.groupby(['cmdb_id','log_name']).agg(
        anomaly_count=('is_anomaly','size'),
        earliest_anomaly_ts_utc8=('ts_utc8','min'),
        max_severity=('severity','max')
    ).reset_index()
    log_anomaly_summary_top20 = log_anomaly_summary.sort_values(by='max_severity', ascending=False).head(20).reset_index(drop=True)
else:
    log_anomaly_summary_top20 = pd.DataFrame(columns=['cmdb_id','log_name','anomaly_count','earliest_anomaly_ts_utc8','max_severity'])

# Keep full filtered tables in kernel: error_window, log_window_filtered, log_joined_window
# Compact outputs to display:
error_window_count, error_counts_top20, error_earliest20, log_anomaly_summary_top20

```
Out[4]:
```
summary = (
    "Summary for 2021-03-07 16:00–16:30 (UTC+8):\n\n"
    "- error_logs:\n"
    "  * Total rows in window: 52\n"
    "  * Top sources: apache02 (26 rows), apache01 (25 rows), Tomcat04 (1 row)\n"
    "  * Earliest entries include Tomcat04 at 2021-03-07 16:13:42+08:00. Many apache error-log lines occur around 16:29 and reference POST /UOCP/... endpoints — indicates heavy frontend traffic/errors on the apache nodes.\n\n"
    "- log.csv anomalies for selected components (Tomcat04, IG02, apache01, apache02, Tomcat01):\n"
    "  * Tomcat04 — log.row_count: anomaly_count=2, earliest=2021-03-07 16:02:00+08:00, max_severity≈1.073\n"
    "  * apache02 — log.row_count: anomaly_count=2, earliest=2021-03-07 16:02:00+08:00, max_severity≈1.050\n"
    "  * Tomcat01 — log.row_count: anomaly_count=2, earliest=2021-03-07 16:24:00+08:00, max_severity≈1.016\n"
    "  * apache01 — log.row_count: anomaly_count=2, earliest=2021-03-07 16:03:00+08:00, max_severity≈1.009\n"
    "  * GC pause maxima reached p95 on Tomcat01 and Tomcat04 (severity = 1.0)\n\n"
    "- Important note: several log.error_count series appear in the anomaly summary but their severity is NaN because the global p95 is zero. That means any non-zero error_count was flagged relative to a zero baseline — this should be interpreted carefully (verify p95 baseline or treat any non-zero error_count as noteworthy).\n\n"
    "Recommendations:\n"
    "  1) Investigate apache01/apache02 immediately (high volume of error logs around 16:29 and elevated log.row_count).\n"
    "  2) Check Tomcat04 and Tomcat01 for increased request/log rates and GC activity.\n"
    "  3) Revisit p95 calculation for log.error_count (zero baseline) to determine correct anomaly semantics for error counts.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(52,     cmdb_id  count
0  apache02     26
1  apache01     25
2  Tomcat04      1,                      ts_utc8   cmdb_id                                            message
0  2021-03-07 16:13:42+08:00  Tomcat04  IG02 POST /UOCP/person/ServiceTest9.json HTTP/...
1  2021-03-07 16:29:03+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest10.jso...
2  2021-03-07 16:29:03+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest1.json...
3  2021-03-07 16:29:05+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
4  2021-03-07 16:29:06+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...
5  2021-03-07 16:29:06+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest3.json...
6  2021-03-07 16:29:07+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest9.json...
7  2021-03-07 16:29:07+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
8  2021-03-07 16:29:07+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest8.json H...
9  2021-03-07 16:29:09+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest8.json H...
10 2021-03-07 16:29:09+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest7.json H...
11 2021-03-07 16:29:09+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
12 2021-03-07 16:29:10+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest4.json H...
13 2021-03-07 16:29:11+08:00  apache01  IPAddress "POST /UOCP/base/ServiceTest6.json H...
14 2021-03-07 16:29:11+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest11.json ...
15 2021-03-07 16:29:11+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest5.json...
16 2021-03-07 16:29:11+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest2.json H...
17 2021-03-07 16:29:13+08:00  apache02  IPAddress "POST /UOCP/base/ServiceTest6.json H...
18 2021-03-07 16:29:13+08:00  apache01  IPAddress "POST /UOCP/person/ServiceTest5.json...
19 2021-03-07 16:29:14+08:00  apache02  IPAddress "POST /UOCP/person/ServiceTest9.json...,     cmdb_id          log_name  anomaly_count  earliest_anomaly_ts_utc8  max_severity
0  Tomcat04     log.row_count              2 2021-03-07 16:02:00+08:00      1.073434
1  apache02     log.row_count              2 2021-03-07 16:02:00+08:00      1.050485
2  Tomcat01     log.row_count              2 2021-03-07 16:24:00+08:00      1.015837
3  apache01     log.row_count              2 2021-03-07 16:03:00+08:00      1.008778
4  Tomcat01  log.gc_pause_max              2 2021-03-07 16:28:00+08:00      1.000000
5  Tomcat04  log.gc_pause_max              2 2021-03-07 16:01:00+08:00      1.000000
6  Tomcat01   log.error_count             31 2021-03-07 16:00:00+08:00           NaN
7  Tomcat04   log.error_count             31 2021-03-07 16:00:00+08:00           NaN
8  apache01   log.error_count             31 2021-03-07 16:00:00+08:00           NaN
9  apache02   log.error_count             31 2021-03-07 16:00:00+08:00           NaN)```
```